##### Content of this Notebook:
- ###### Operations (1-5): Reading, Select, Alias, Filter, withColumnRenamed, withColumn
- ###### Operations (6-10): withColumn, Typecast, Order By/Sort, Limit, Drop
- ###### Operations (11-15): Drop Duplicate, Union/unionByName,String Function, Date Function, DateDiff 
- ###### Operations (16-20): Date Format, Handling Nulls, Split & Indexing, Explode, Array Contains
- ###### Transformation (21-23): Group By, Collect List, Pivot

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
df = spark.read.table('workspace.default.big_mart_sales')

In [0]:
%sql
select * from big_mart_sales

![image_1784307527207.png](./image_1784307527207.png "image_1784307527207.png")

###### Read player.csv file, and do the DDL Schema operation on it:

In [0]:
playerDf = spark.read.format('csv')\
        .option('inferSchema',True)\
        .option('header',True)\
        .load('/Workspace/Users/official.sushantshanu@gmail.com/Drafts/IPL-Analysis/Raw_Data/Player.csv')

In [0]:
playerDf.display()

In [0]:
playerDf.printSchema()

In [0]:
playerDDLSchema = '''
                          PLAYER_SK INTEGER,
                          Player_Id STRING,
                          Player_Name STRING,
                          DOB DATE,
                          Batting_hand STRING,
                          Bowling_skill STRING,
                          Country_Name STRING
                  '''

In [0]:
## Reading as per DDL Schema:
playerDfDDL = spark.read.format('csv')\
        .schema(playerDDLSchema)\
        .option('header',True)\
        .load('/Workspace/Users/official.sushantshanu@gmail.com/Drafts/IPL-Analysis/Raw_Data/Player.csv')
playerDfDDL.display()

###### Reading the same Data now with StructType() Schema:

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_struct_schema = StructType ([
                        StructField('PLAYER_SK', IntegerType(), True),
                        StructField('Player_Id', StringType(), True),
                        StructField('PLAYER_NAME', StringType(), True),
                        StructField('DOB', DateType(), True),
                        StructField('Batting_hand', StringType(), True),
                        StructField('Bowling_skill', StringType(), True),
                        StructField('Country_Name', StringType(), True)
])

In [0]:
playerDfStruct = spark.read.format('csv')\
        .schema(my_struct_schema)\
        .option('header',True)\
        .load('/Workspace/Users/official.sushantshanu@gmail.com/Drafts/IPL-Analysis/Raw_Data/Player.csv')
playerDfStruct.display()
playerDf.printSchema()
playerDfStruct.printSchema()
playerDf.columns
playerDfStruct.columns

###### Reading Data:

#### PySpark Lesson:

###### Operation 1: Reading the Data: Big Mart Sales

In [0]:
## Find out the files present in the path
dbutils.fs.ls('/FileStore/')

In [0]:
## In my case, i've created table from a csv file, so in my case, i'm directly creating df from the table

# df = spark.read.format('csv')\
#     .option('inferSchema',True)\
#     .option('header',True)\
#     .load('workspace.default.big_mart_sales')

df = spark.read.table('workspace.default.big_mart_sales')

In [0]:
df.display()
df.printSchema()

###### Operation 2: SELECT

In [0]:
## Method 1:
df.select('Item_Identifier','Item_Weight','Item_Fat_Content').display()

In [0]:
## Method 2:
df.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content')).display()

###### Operation 3: ALIAS

In [0]:
df.select(col("Item_Identifier").alias("Item_ID")).display()

###### Operation 4: FILTER

In [0]:
## Scenario 1: Full the DF and get the Item Fat Content == Regular
df.filter(col("Item_Fat_Content")=='Regular').display()

In [0]:
## Scenario 2: Records which have Item Type == Soft Drinks & Item Weight < 10
df.filter((col("Item_Type")=='Soft Drinks') & (col("Item_Weight")<10)).display()

In [0]:
## Scenario 3: Data with Tier in Tier 1 or 2 (function - isin()) and Outlet Size is Null (function - isNull())
df.filter((col("Outlet_Location_Type").isin("Tier 1","Tier 2")) & (col("Outlet_Size").isNull())).display()

###### Operation 5: withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt').display()

###### Operation 6: withColumn - Create the new column/modify the column

In [0]:
## Scenario 1.1: Create new column - Here we are using lit() function to have the constant value we want to give to that column
df = df.withColumn('flag',lit('new'))
df.display()

In [0]:
## Scenario 1.2: Create new column - New columns needs some transformation for it's value
df.withColumn('Mutiply_Result',(col('Item_Weight')*col('Item_MRP'))).display()

In [0]:
## Scenario 2: Change the value ofexisting column by transformation using regexp_replace()
df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
    .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","Low F"))\
    .display()

###### Operation 7: Type Casting- Change the data type to another

In [0]:
df = df.withColumn('Item_Weight',col('Item_Weight').cast(StringType()))
df.printSchema()

###### operation 8: Sort/Order By

In [0]:
## Scenario 1: Sort the dataframe base on Item Weight in Desc order
df.sort(col('Item_Weight').desc()).display()

In [0]:
## Scenario 3: Sorting based on multiple columns || For sorting we can give either True/False or 1/0 as it accepts boolean value
df.sort(['Item_Weight','Item_Visibility'],ascending=[False,False]).display()

###### Operation 9: Limit

In [0]:
 df.limit(3).display()

###### Operation 10: Drop

In [0]:
df.drop('Item_Visibility','Item_Type').display()

###### Operation 11: Drop Duplicates

In [0]:
df.dropDuplicates().display()

In [0]:
## Scenario 2: Delete dupluicates based on subset of column
df.drop_duplicates(subset=['Item_Type']).display()

In [0]:
df.distinct().display()

###### Operation 12: Union & Union By Name

In [0]:
 data1 = [('1','Kad'),
         ('2','Sid')]
 schema1 = 'id STRING', 'name STRING'

 data2 = [('3','Rahul'),
          ('4','Jas')]
 schema2 = 'id STRING', 'name STRING'

 df1 = spark.createDataFrame(data1,schema1)
 df2 = spark.createDataFrame(data2,schema2)

In [0]:
df1.display()
df2.display()

In [0]:
## Scenario 1: Union
df1.union(df2).display()

In [0]:
data3 = [('Kad','1'),
         ('Sid','2')]
schema3 =  'name STRING', 'id STRING'

data4 = [('Kad','1'),
         ('Kad','2')]
schema4 =  'name STRING', 'id STRING'

df3 = spark.createDataFrame(data3,schema3)
df3.display()
df4 = spark.createDataFrame(data4,schema4)
df4.display()

In [0]:
df3.union(df2).display()
## Hence, it does not identify and create a mess in the data, for that we use unionByName() - it itself creates the mapping as per the name between two df
df3.unionByName(df2).display()

In [0]:
## FAILED!!
# df3.union(df4).display() ## without duplicates
# df3.unionAll(df4).display() ## with duplicates

###### Operation 13: String Functions

In [0]:
## Scenarion 1: Initcap()
df.select(initcap('Item_Type')).display()

In [0]:
## Scenarion 2: Upper & Lower case
df.select(upper('Item_Type')).display()
df.select(lower('Item_Type')).display()

###### Operation 14: Date Functions - Current_Date(), Date_Add(), Date_Sub()

In [0]:
df = df.withColumn('Current_Date',current_date())
df.display()

In [0]:
df = df.withColumn('One_Week_Later',date_add(col('Current_Date'),7))\
    .withColumn('One_Week_Before',date_sub(col('Current_Date'),7))\
    .withColumn('One_Month_Before',date_add(col('Current_Date'),-30))
df.display()

###### Operation 15: DateDiff

In [0]:
df = df.withColumn('Date_Difference',datediff('One_Week_Later','One_Week_Before'))
df.display()

###### Operation 16: Date Format

In [0]:
df = df.withColumn('One_Month_Before',date_format('One_Month_Before','dd-MM-yyyy'))
df.display()

###### Operation 17: Handling Nulls

In [0]:
## Strategy 1: Dropping Nulls || Any - Drop record which have null in any of the column, All - Drop only those records which have null in all of its columns
df.dropna('all').display()
df.dropna('any').display()

In [0]:
## Strategy 2: Drop those records which is having null vlaue in Outlet size column - using subset
df.dropna(subset=['Outlet_Size']).display()

In [0]:
## Type casting Item Weight to String so the fill na can work
df = df.withColumn('Item_weight',col('Item_Weight').cast(StringType()))
df.display()

In [0]:
## Strategy 3: Fill the null value with some value
df.fillna('NotAvailable').display()

In [0]:
df.fillna('NotAvailable',subset=['Item_Weight']).display()

##### Operation 18: Split & Indexing

In [0]:
## Split Function
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).display()

In [0]:
## Index Function
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).display()


##### Operation 19: Explode

In [0]:
df = df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df.display()

In [0]:
## Implemting explode function-
df.withColumn('Outlet_Type',explode('Outlet_Type')).display()

###### Operation 20: Array Contains

In [0]:
df.withColumn('Type1_Flag',array_contains('Outlet_Type','Type1')).display()

###### Transformation 21: Group By

In [0]:
df.groupBy('Item_Type').agg((sum('Item_MRP')).alias('Total Sales')).display()

In [0]:
df.groupBy('Item_Type').agg((avg('Item_MRP')).alias('Average Sales Value')).display()

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg((avg('Item_MRP')).alias('Average Sales Value'))\
    .sort(col('Item_Type').desc())\
    .display()

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg((sum('Item_MRP')).alias('Total Sales Value'),(avg('Item_MRP')).alias('Average Sales Value'))\
    .sort(col('Item_Type').desc())\
    .display()

###### Transformation 22: Collect_list - Collect all the value in a list

In [0]:
 data1 = [('user1','Book1'),
         ('user1','Book2'),
         ('user2','Book2'),
         ('user2','Book4'),
         ('user3','Book1')]
 schema1 = 'user', 'book'
 df1 = spark.createDataFrame(data1,schema1)
 df1.display()

In [0]:
df1.groupBy('user').agg(collect_list('book')).display()

###### Transformation 23: Pivot:

In [0]:
df_pivot = df.groupBy('Item_Type').pivot('Outlet_Size').agg(sum('Item_MRP'))
df_pivot.display()